In [ ]:
# python setup.py install
# pip install fasttext openpyxl pyreadstat utils xlrd codauto

In [1]:
import pandas as pd
import os
import sys
import re
import pickle
from codauto import Structured, StructuredWZ
from codauto import Codifier
from codauto import CodifierFastText
#sys.path.append("/home/onyxia/work/WP10_Cluster1_StatCodGen/codauto")
#from structured import Structured, StructuredWZ
#from codifier import Codifier
#from codifier_fasttext import CodifierFastText

# German WZ Classification

In [3]:
root_path = './test'
language = 'de'

## Using Structured for German NACE

In [4]:
# Load the structure downloaded from the link provided earlier
path_struc_wz = r'./../resources'#/WP10_Cluster1_StatCodGen/de/wz_classification/' # Change the path to the location of the downloaded file
name_stuc_wz = 'WZ_2025-DE-2026-01-27-Gliederung.xlsx'
structure_wz_df = pd.read_excel(
    os.path.join(path_struc_wz, name_stuc_wz),
    sheet_name=1,
    dtype=str,
    skiprows=0,
    header=0,
    usecols=[0,2]
)
# For any classification, a DataFrame with two columns must be used: the first with the classification codes and the second with the titles.
# The codes must be provided in hierarchical order, meaning that parent classes must appear before their children.
structure_wz_df.head()

/opt/python/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,Schlüssel WZ 2025,Titel
0,A,"Land- und Forstwirtschaft, Fischerei"
1,01,"Landwirtschaft, Jagd und damit verbundene Täti..."
2,01.1,Anbau ein- und zweijähriger Pflanzen
3,01.11,Anbau von anderem Getreide als Reis sowie von ...
4,01.11.0,Anbau von anderem Getreide als Reis sowie von ...


In [5]:
# Generate the instance of StructuredWZ
wz_structure = StructuredWZ(
    structure_df=structure_wz_df,
    names_l=[
    'Section',
    'Division',
    'Group',
    'Class',
    'Subclass'
    ] # Names of the levels ordered from highest to lowest hierarchy
# As a best practice, it is recommended to use underscores instead of spaces and to avoid accents
)
wz_structure.level_dict

{0: 'Section', 1: 'Division', 2: 'Group', 3: 'Class', 4: 'Subclass'}

In [ ]:
# Inspect the structure
#wz_structure.get_level('01.1')

In [ ]:
#wzs = wz_structure.load_structue(structure_wz_df)
#wzs[wzs['level'] == 1]

## Using Codifier for German NACE

In [6]:
# WZ test data
train_df = pd.DataFrame({'label': ['35122', '78202', '23200', '71126']*20,
                         'text': ['text1', 'text2', 'word1', 'word2']*20}) 
test_df = pd.DataFrame({'label': ['35122', '78202']*20, 
                        'text': ['tekst3', 'tekstt2']*20, 
                        'source': ['s1', 's1']*20}) 

In [7]:
wz_codifier = CodifierFastText(
    root_path=root_path,
    structure_instance=wz_structure,
    train_df=train_df,
    test_df=test_df
    ,language=language
)

2026-02-10 10:06:26,319 - INFO - Loading train_df..
2026-02-10 10:06:26,320 - INFO - train_df raw data count: 80


AttributeError: 'CodifierFastText' object has no attribute 'language'

In [ ]:
wz_codifier.train()

In [ ]:
# Example prediction
desc_l=[
    'Frisör',
    'Haarschneider'
       ]

mode = 'assistance'
hierarchical_level = 5
threshold = 0.03

wz_codifier.predict(
    desc_l=desc_l,
    mode=mode,
    hierarchical_level=hierarchical_level,
    threshold=threshold
)

In [ ]:
wz_codifier.evaluate()